# Data Reliability - 可观测性 (Observability)

> **适用场景**: 数据平台监控、SLA 保障、故障发现
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频

## 目录
1. Data Lineage 体系
2. SLA vs SLO vs SLI
3. Monitoring Metrics 设计
4. Alert Fatigue 避免策略
5. 练习题

---
## 1. Data Lineage 体系

### 什么是数据血缘？
数据血缘（Data Lineage）记录数据从**源头到消费的完整流转路径**，回答：
- 这张表的数据从哪里来？
- 这个字段经过了哪些转换？
- 如果 orders 表出问题，哪些下游报表会受影响？

### 血缘的层次

```
列级血缘（最细粒度）
  ↓
表级血缘
  ↓
系统/Pipeline 级血缘（最粗粒度）
```

**示例血缘图**：
```
MySQL (raw.orders)
    ↓ Fivetran
BigQuery (raw.orders)
    ↓ dbt (stg_orders)
BigQuery (staging.stg_orders)
    ↓ dbt (fct_orders)
BigQuery (analytics.fct_orders)
    ↓                    ↓
Tableau Dashboard    ML Feature Store
```

### 血缘采集方式

**1. 解析 SQL/代码（静态分析）**
```python
# sqlglot 解析 SQL 获取血缘
import sqlglot

sql = """
INSERT INTO analytics.fct_orders
SELECT o.order_id, c.name
FROM staging.stg_orders o
JOIN staging.stg_customers c ON o.customer_id = c.id
"""

lineage = sqlglot.lineage('analytics.fct_orders', sql)
# 输出: fct_orders ← [stg_orders, stg_customers]
```

**2. dbt 内置血缘**
```bash
# 生成血缘图
dbt docs generate
dbt docs serve  # 在浏览器查看血缘 DAG

# 找到影响 fct_orders 的所有上游模型
dbt ls --select +fct_orders

# 找到 fct_orders 影响的所有下游
dbt ls --select fct_orders+
```

**3. OpenLineage（开源标准）**
```python
# OpenLineage 事件格式
{
  'eventType': 'COMPLETE',
  'job': {'name': 'dbt.fct_orders'},
  'inputs': [{'name': 'stg_orders'}, {'name': 'stg_customers'}],
  'outputs': [{'name': 'fct_orders'}]
}
# Airflow、Spark、dbt 均支持 OpenLineage
```

### 血缘的应用
- **Impact Analysis**：上游变更前评估影响范围
- **Root Cause Analysis**：数据问题溯源
- **数据治理**：PII 字段在哪里被使用？
- **优化**：发现未被使用的表（可以删除）

---
## 2. SLA vs SLO vs SLI

### 三个概念

| 概念 | 全称 | 定义 | 面向对象 |
|------|------|------|----------|
| **SLI** | Service Level Indicator | 实际测量的指标值 | 内部 |
| **SLO** | Service Level Objective | 内部目标（比 SLA 更严格）| 内部 |
| **SLA** | Service Level Agreement | 对外承诺，违反有惩罚 | 对外（合同）|

### 数据工程 SLI 示例
```
SLI = 实际测量值
├── 数据新鲜度：MAX(event_time) 距 NOW() 的分钟数
├── Pipeline 成功率：过去7天成功运行次数 / 总运行次数
├── 查询 p99 延迟：第99百分位查询响应时间
└── 数据完整性：实际行数 / 预期行数
```

### 实际层级关系
```
SLA（对外）：报表数据每天 8 AM 前可用
    ↓ 更严格
SLO（内部目标）：数据每天 7:30 AM 前完成处理
    ↓ 实际测量
SLI（指标）：`SELECT MAX(processed_at) FROM daily_report`
```

### Error Budget
```
SLO = 99.9% 可用性（1个月 = 720小时）
Error Budget = (1 - 99.9%) × 720小时 = 0.72小时 ≈ 43分钟

若本月已消耗 30 分钟 → 剩余 13 分钟 budget
→ 暂停新功能发布，优先保稳定性
```

---
## 3. Monitoring Metrics 设计

### 数据平台核心监控指标

**Pipeline 层监控**
```python
# Airflow metrics（可导出到 Datadog/Prometheus）
# - dag_run.duration: DAG 运行时长
# - dag_run.failed: 失败次数
# - task_instance.duration: Task 运行时长
# - scheduler_heartbeat: Scheduler 心跳

# 自定义 Airflow callback
def on_failure_callback(context):
    dag_id = context['dag'].dag_id
    task_id = context['task_instance'].task_id
    send_slack_alert(f'❌ {dag_id}.{task_id} 失败')
    push_metric('pipeline.failure', 1, tags=[f'dag:{dag_id}'])
```

**数据层监控**
```sql
-- 建立监控元数据表
CREATE TABLE data_quality_metrics (
    table_name STRING,
    metric_name STRING,          -- 'row_count', 'null_rate', 'freshness_minutes'
    metric_value FLOAT64,
    threshold_warn FLOAT64,
    threshold_error FLOAT64,
    status STRING,               -- 'ok', 'warn', 'error'
    checked_at TIMESTAMP
);

-- 每次质量检查后插入结果
INSERT INTO data_quality_metrics
SELECT
    'orders' AS table_name,
    'freshness_minutes' AS metric_name,
    TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), MAX(created_at), MINUTE) AS metric_value,
    30 AS threshold_warn,
    60 AS threshold_error,
    CASE
        WHEN TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), MAX(created_at), MINUTE) > 60 THEN 'error'
        WHEN TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), MAX(created_at), MINUTE) > 30 THEN 'warn'
        ELSE 'ok'
    END AS status,
    CURRENT_TIMESTAMP() AS checked_at
FROM orders;
```

### 监控仪表板设计
```
数据平台健康度 Dashboard
├── Pipeline Success Rate（过去24小时）
├── Data Freshness（关键表的最新数据时间）
├── Quality Score（各表质量指标通过率）
├── Processing Latency Trend（P50/P95/P99）
├── Error Budget Burn Rate
└── Active Incidents
```

---
## 4. Alert Fatigue 避免策略

### 什么是 Alert Fatigue？
过多、过频、不重要的告警导致 on-call 人员**对告警脱敏**，真正重要的告警被忽视。

### 常见原因
- 告警阈值设置过于敏感（任何偏差都告警）
- 没有优先级区分（P1 和 P3 用同样的告警方式）
- 同一问题反复告警（没有去重/聚合）
- 告警没有行动指导（不知道怎么处理）

### 解决策略

**策略 1：分级告警**
```
P1（立刻响应，PagerDuty 呼叫）:
  - 核心报表延迟 > 2小时
  - 数据丢失 > 5%
  - 生产 pipeline 连续失败 3次

P2（工作时间响应，Slack 通知）:
  - 数据延迟 30-60 分钟
  - 单次 pipeline 失败

P3（每日汇总邮件）:
  - 非关键表质量下降
  - 慢查询告警
```

**策略 2：告警去重和抑制**
```yaml
# PagerDuty / Alertmanager 去重
# 同一 DAG 失败，15分钟内只发一次告警
- alert: PipelineFailure
  for: 5m            # 持续5分钟才告警（避免瞬时抖动）
  group_wait: 30s
  group_interval: 15m  # 同一组告警最少间隔15分钟
  repeat_interval: 4h  # 持续问题每4小时提醒一次
```

**策略 3：有意义的告警内容**
```python
def format_alert(context):
    return {
        'title': f'❌ {dag_id} 失败',
        'severity': 'P2',
        'impact': '用户无法看到昨日订单报表',
        'probable_cause': '上游 orders 表延迟（最近记录: 2小时前）',
        'runbook': 'https://wiki/runbooks/orders-pipeline',
        'action': '1. 检查 Airflow logs\n2. 查看上游 pipeline 状态'
    }
# 好的告警 = 标题 + 影响 + 原因 + 处理步骤 + Runbook 链接
```

**策略 4：动态阈值（Anomaly Detection）**
```python
# 不用固定阈值，用历史数据的统计学阈值
# 当前值距离历史均值超过 2.5 个标准差才告警
# 工具：Monte Carlo、Anomalo（商业）或自建
threshold = mean + 2.5 * std_dev
if current_value < threshold * 0.8:  # 低于历史均值 80%
    send_alert()
```

**策略 5：定期 Alert Review**
- 每月统计告警量和响应情况
- 消除过去3个月内从未触发行动的告警
- 降级只是"记录"而非"通知"的低价值告警

---
## 5. 练习题

### Q1 [高频] SLA、SLO、SLI 分别是什么？以数据平台为例说明？

<details><summary>参考答案</summary>

- **SLI**（指标）：实际测量值。例：`SELECT TIMESTAMP_DIFF(NOW(), MAX(updated_at), MINUTE) FROM daily_report`，当前值为 25 分钟
- **SLO**（内部目标）：内部设定的目标。例：数据延迟 SLO = 每天 8AM 前完成（99.5% 的工作日）
- **SLA**（对外承诺）：写入合同。例：商业 BI 客户的 SLA = 每天 9AM 前数据可用，违反则退款

**层级关系**：SLO 比 SLA 更严格（内部 buffer），给工程团队留有修复时间而不违反 SLA。
</details>

---

### Q2 [高频] 数据血缘有哪些实际应用场景？

<details><summary>参考答案</summary>

1. **影响分析（Impact Analysis）**：上游 schema 变更前，通过血缘找到所有下游依赖，评估风险
2. **根因分析（RCA）**：报表数据错误，通过血缘向上追溯找到问题数据源
3. **PII 合规**：追踪个人数据字段流转路径，保证 GDPR 合规（用户删除时知道要清理哪些地方）
4. **无用数据清理**：找到没有下游消费者的表/字段，安全删除节省成本
5. **复杂查询优化**：了解数据流向后，可以在合适的位置加缓存或物化视图
6. **文档和 Onboarding**：新工程师快速理解数据平台架构
</details>

---

### Q3 如何避免 Alert Fatigue？团队告警过多时你会怎么做？

<details><summary>参考答案</summary>

**诊断**：
1. 统计过去1个月告警量、告警类型分布、响应率
2. 识别哪些告警被忽略/静音了

**治理**：
1. **分级**：只有 P1 才 PagerDuty 夜间呼叫，P2/P3 走 Slack
2. **去重**：同一问题 15 分钟内合并一条
3. **内容改进**：每条告警必须包含影响、可能原因、处理步骤
4. **动态阈值**：用 z-score 替代固定阈值，减少误报
5. **定期清理**：每月 Alert Review，删除无意义告警
6. **Runbook**：每个告警对应 Runbook，减少处理时间和心理负担
</details>

---

### Q4 设计数据平台的核心监控指标体系，至少列出5类指标？

<details><summary>参考答案</summary>

1. **Pipeline 可靠性**：
   - 成功率（过去7天 DAG 成功率）
   - 平均运行时长 & P95 运行时长
   - 失败后自动恢复率

2. **数据新鲜度**：
   - 关键表距最新数据的延迟（分钟）
   - SLA 达标率（按时完成的比例）

3. **数据质量**：
   - 质量检查通过率（dbt test pass rate）
   - 关键字段非空率
   - 行数异常检测告警次数

4. **系统资源**：
   - 仓库计算成本（BigQuery slot 使用量）
   - 存储增长趋势
   - 查询 P99 延迟

5. **Incident 指标**：
   - MTTD（Mean Time to Detect，平均检测时间）
   - MTTR（Mean Time to Restore，平均恢复时间）
   - Error Budget 消耗率
</details>